In [3]:
import re
from collections import defaultdict, Counter

In [4]:
reviews = [
    "Absolutely loved it! Exceeded my expectations.",
    "Fantastic service and great attention to detail.",
    "I will definitely come back again.",
    "Five stars—couldn’t be happier!",
    "Everything worked perfectly from start to finish.",
    "It was a smooth and enjoyable experience.",
    "The product quality is top-notch.",
    "I’m really impressed with the level of professionalism.",
    "Clean, efficient, and reliable.",
    "Excellent value for money.",
    "The customer support team was very helpful.",
    "Quick delivery and the item matched the description.",
    "The interface was intuitive and easy to use.",
    "I’ve already recommended it to my friends.",
    "High quality and affordable.",
    "Surprisingly good, exceeded all my expectations.",
    "Super friendly staff and fast response times.",
    "Setup was easy and everything worked immediately.",
    "The packaging was beautiful and secure.",
    "Very satisfied with my purchase.",
    "This was one of the best decisions I’ve made!",
    "I felt well taken care of throughout the process.",
    "The colors were vibrant and exactly as pictured.",
    "Reliable and consistent every single time.",
    "I love it—will definitely buy again.",
    "Completely disappointed—nothing worked.",
    "Terrible experience from start to finish.",
    "Would not recommend to anyone.",
    "Poor quality and overpriced.",
    "I had high hopes but it failed miserably.",
    "Customer service was rude and unhelpful.",
    "The instructions were unclear and confusing.",
    "It broke within the first week.",
    "Slow delivery and bad packaging.",
    "Not worth the money at all.",
    "App kept crashing every few minutes.",
    "The item looked nothing like the photos.",
    "Misleading description and poor build quality.",
    "Very frustrating experience overall.",
    "It didn’t meet even basic expectations.",
    "I had to return it the same day I got it.",
    "Overhyped and underdelivered.",
    "I regret making this purchase.",
    "There were bugs they never fixed.",
    "The support team didn’t respond to my emails.",
    "It’s just a waste of time and money.",
    "Looks cheap and feels worse.",
    "The service was shockingly bad.",
    "Error messages every time I tried to use it.",
    "Nothing worked as advertised.",
    "The quick brown fox jumps over the lazy dog.",
    "The quick brown bear sleeps.",
    "The quick brown fox sleeps",
]

### tokenization funcs

In [5]:
def normalize(text):
  text = text.lower()
  text = re.sub(r"\W", '', text)
  text = re.sub(r"\s+", '', text)
  return text

def tokenize(text, special_tokens = True):
  # TODO: exclude multi-word
  tokens = re.split('[—\-\s\']', text)
  tokens = list(map(normalize, tokens))

  # This one is optional, depending on your use case
  if special_tokens:
    tokens = ["<s>"] + list(map(normalize, tokens)) + ["</s>"]
  return tokens

<>:9: SyntaxWarning: invalid escape sequence '\-'
<>:9: SyntaxWarning: invalid escape sequence '\-'
C:\Users\ayhec\AppData\Local\Temp\ipykernel_6288\3083130683.py:9: SyntaxWarning: invalid escape sequence '\-'
  tokens = re.split('[—\-\s\']', text)


### ngram

In [6]:
def ngram(tokens, n = 3):
  ngrams = Counter()

  for i in range(len(tokens)):
    grams = tokens[i] if n == 1 else tuple(tokens[i:i+n])
    ngrams[grams] += 1

  return ngrams

In [7]:
tokens = []

for review in reviews:
  tokens += tokenize(review)

trigrams = ngram(tokens)
bigrams = ngram(tokens, 2)
unigrams = ngram(tokens, 1)

vocab_size = len(set(tokens))

(bigrams)

Counter({('</s>', '<s>'): 52,
         ('<s>', 'the'): 12,
         ('<s>', 'i'): 6,
         ('expectations', '</s>'): 3,
         ('<s>', 'it'): 3,
         ('the', 'quick'): 3,
         ('quick', 'brown'): 3,
         ('my', 'expectations'): 2,
         ('will', 'definitely'): 2,
         ('again', '</s>'): 2,
         ('everything', 'worked'): 2,
         ('from', 'start'): 2,
         ('start', 'to'): 2,
         ('to', 'finish'): 2,
         ('finish', '</s>'): 2,
         ('money', '</s>'): 2,
         ('support', 'team'): 2,
         ('delivery', 'and'): 2,
         ('the', 'item'): 2,
         ('to', 'use'): 2,
         ('to', 'my'): 2,
         ('quality', 'and'): 2,
         ('<s>', 'very'): 2,
         ('purchase', '</s>'): 2,
         ('nothing', 'worked'): 2,
         ('i', 'had'): 2,
         ('service', 'was'): 2,
         ('it', '</s>'): 2,
         ('brown', 'fox'): 2,
         ('sleeps', '</s>'): 2,
         ('<s>', 'absolutely'): 1,
         ('absolutely', 'loved'):

### Frequency Table

In [8]:
def create_freq_table(grams, n = 3):
  freq = defaultdict(Counter) if n > 1 else Counter(grams)

  if n == 1:
    return freq

  for gram in grams:
    prefix = tuple(gram[:-1])
    next_word = gram[-1]

    freq[prefix][next_word] += 1

  return freq

In [25]:
trigram_freq = create_freq_table(trigrams)
bigram_freq = create_freq_table(bigrams, 2)
unigram_freq = create_freq_table(unigrams, 1)

### Prediction function

#### Frequency Table

In [10]:
def predict_next(word, model, top_k=3):
    if (isinstance(model, Counter)):
      return model.most_common(top_k)

    next_words = model[tuple(word.lower().split(" "))]
    return next_words.most_common(top_k)

In [11]:
predict_next("quick brown", trigram_freq)

[('fox', 1), ('bear', 1)]

In [12]:
predict_next("quick", bigram_freq)

[('delivery', 1), ('brown', 1)]

In [13]:
predict_next("brown", unigram_freq)

[('<s>', 53), ('</s>', 53), ('the', 22)]

#### Markov Assumption

In [14]:
import math

def predict_next(word, model, vocab_size, top_k=3):
    if isinstance(model, Counter):
        total = sum(model.values())
        probabilities = [
            (w, math.log((count + 1) / (total + vocab_size)))
            for w, count in model.items()
        ]
        probabilities.sort(key=lambda x: x[1], reverse=True)
        return probabilities[:top_k]

    word = tuple(word.lower().split(" "))
    if word not in model:
        return []
    next_words = model[word]
    total = sum(next_words.values())

    probabilities = [(w, math.log((count + 1) / (total + vocab_size))) for w, count in next_words.items()]
    probabilities.sort(key=lambda x: x[1], reverse=True)

    return probabilities[:top_k]

In [15]:
trigram_candidates = predict_next("The quick", trigram_freq, vocab_size)
trigram_candidates

[('brown', -4.584967478670572)]

In [16]:
bigram_candidates = predict_next("quick", bigram_freq, vocab_size)
bigram_candidates

[('delivery', -4.590056548178043), ('brown', -4.590056548178043)]

In [17]:
unigram_candidates = predict_next("lazy", unigram_freq, vocab_size)
unigram_candidates

[('<s>', -2.4614863755799017),
 ('</s>', -2.4614863755799017),
 ('the', -3.3149762062150265)]

### Interpolation

In [18]:
def interpolate(w1, w2, w3, weights = (0.1, 0.3, 0.6)):
  # P(w₃ | w₁, w₂) = λ₁ × P(w₃) + λ₂ × P(w₃ | w₂) + λ₃ × P(w₃ | w₁, w₂)

  # P(w₃)
  prob1 = unigrams.get(w3) / sum(unigrams.values()) if w3 in unigrams else 0

  # P(w₃ | w₂)
  prob2 = bigrams.get((w2, w3)) / unigrams.get(w2) if (w2, w3) in bigrams else 0

  # P(w₃ | w₁, w₂)
  prob3 = trigrams.get((w1, w2, w3)) / bigrams.get((w1, w2)) if (w1, w2, w3) in trigrams else 0

  return (prob1 * weights[0]) + (prob2 * weights[1]) + (prob3 * weights[2])

def predict_with_interpolation(text):
  parsed = tokenize(text, False)
  w1, w2 = parsed[-2:]

  probabilities = Counter()

  for token in set(tokens):
    probabilities[token] = interpolate(w1, w2, token)

  return probabilities.most_common(5)

predict_with_interpolation("the quick")

[('brown', 0.8256849315068493),
 ('delivery', 0.07545662100456621),
 ('</s>', 0.012100456621004566),
 ('<s>', 0.012100456621004566),
 ('the', 0.00502283105022831)]

### Perplexity

#### Normal

In [19]:
def perplexity(sentence, model):
  tokens = tokenize(sentence)
  bigrams = ngram(tokens)

  total_prob = 1.0
  word_count = len(tokens)

  for bigram in bigrams:
    prefix = tuple(bigram[:-1])
    total = 0
    count = 0
    prob = 1e-6

    if prefix in model:
      total = sum(model[prefix].values())
      count = model[prefix].get(bigram[-1], 0)
      prob = (count + 1) / (total + 1)

    total_prob *= prob

  return total_prob ** -(1 / word_count)

In [20]:
perplexity("wqasdasd asdasds asd", trigram_freq)

63095.734448019364

In [21]:
perplexity("The quick brown fox", trigram_freq)

17.07114038846183

#### With Log

In [22]:
import math

def perplexity(sentence, model):
  tokens = tokenize(sentence)
  bigrams = ngram(tokens)

  log_prob = 0.0
  word_count = len(tokens)

  for bigram in bigrams:
    prefix = tuple(bigram[:-1])
    total = 0
    count = 0
    prob = 1e-6

    if prefix in model:
      total = sum(model[prefix].values())
      count = model[prefix].get(bigram[-1], 0)
      prob = (count + 1) / (total + 1)

    log_prob += math.log(prob)

  ave = log_prob / word_count
  return math.exp(-ave)

In [23]:
perplexity("wqasdasd asdasds asd", trigram_freq)

63095.73444801934

In [24]:
perplexity("The quick brown fox", trigram_freq)

17.07114038846183